In [1]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# import torch

# # Set device to GPU with index 1
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print("Using device:", device)
# print(torch.cuda.get_device_name(device))

In [2]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModel, T5EncoderModel
import re
import numpy as np
from tqdm.auto import tqdm

/home/prabhat/miniconda3/envs/abcp_finder/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def get_peptide_embeddings(sequences, model_name, device):
    """
    Generates peptide embeddings using a pretrained transformer model.

    Args:
        sequences (list): A list of peptide sequences (e.g., ['GAVW', 'LLNQELLLNPTHQIYPV']).
        model_name (str): The name of the Hugging Face model to use.
        device (str): 'cuda' or 'cpu'.

    Returns:
        np.array: A NumPy array of peptide embeddings.
    """
    
    # 1. Load the tokenizer and model
    if 't5' in model_name.lower():
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = T5EncoderModel.from_pretrained(model_name).to(device)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)
    
    print(model.eval())
    all_embeddings = []

    # 2. Pre-process sequences
    # Add a space between each amino acid for ProtBERT and ProtT5
    sequences_processed = [" ".join(list(re.sub(r"[UZOB]", "X", seq))) for seq in sequences]
    
    for seq in tqdm(sequences_processed, desc=f"Generating embeddings with {model_name}"):
        # 3. Tokenize and encode
        ids = tokenizer.batch_encode_plus([seq], add_special_tokens=True, padding=True, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = model(**ids)
        
        # 4. Get the last hidden state (the main embedding layer)
        # This gives you a per-residue embedding for each amino acid in the sequence.
        last_hidden_state = outputs.last_hidden_state
        
        # 5. Pool the per-residue embeddings to get a single peptide embedding
        # We take the mean of the embeddings for all amino acids in the peptide.
        # This is a common and effective pooling strategy.
        embedding = last_hidden_state[0, 1:-1].mean(dim=0).cpu().numpy()
        all_embeddings.append(embedding)

    return np.array(all_embeddings)


In [5]:
# ABCP_tokenisation
with open("ABCP_train.fasta") as abcp:
    abcp_lines = [line.strip("\n") for line in abcp.readlines()[1::2]]

abcp_lines

['KAAKKAWKWAKKAAKWAKKAA',
 'KKKFPWWWPFKKKCKKKFPWWWPFKKKC',
 'ILPWKWPWWPWRR',
 'GLFLDTLKGAAKDVAGKLEGLKCKITGCKLP',
 'ALWKDILKNVGKAAGKAVLNTVTDMVNQ',
 'WRRRYRRWRRRRRWRRRPRR',
 'GLFDIVKKIAGHIVSSI',
 'FIHHIIGGLFSAGKAIHRLIRRRRR',
 'KAAKKAWKAWKKAAKAAWKKAA',
 'FKLAFKLAKKAFL',
 'VAEAREELERLEARLGQARGELKKWKMRRNQFWLKLQR',
 'GCRRLCWKQRCVTYCRGR',
 'ETFSDWWKLLAE',
 'GILSSIKGVAKGVAKNVAAQLLDTLKCKITGC',
 'FLSGIVGMLAKLFKFLKALMFLSGIVG',
 'AAKKWAKAKWAKAKKWAKAA',
 'FLGMIPKLIKKLIKAFK',
 'FAKKALKALKKL',
 'GRRKRKWLRRIGKGVKIIGGAALDHL',
 'PAWRKAFRWAWRMKKLAA',
 'KLALKLALKALKAAKLA',
 'GLFDVIKKVASVIKKL',
 'NYPQRPCRGDKGPDC',
 'LRPEIRYAQELRRIGDEFNE',
 'FRRFFKWFRRFFKFF',
 'FLSGIVAMLGKLF',
 'ATCETPSKHFNGLCIRSSNCASVCHGEHFTDGRCQGVRRRCMCLKPC',
 'FLSTIWNGIKSLL',
 'FPLPCAYKGTYC',
 'GLFDIVKKVVGAFGSL',
 'GIGKFLKKAKKFGKAFVKILKK',
 'KWLRRVWRWWR',
 'GLLDLLKLLLKAAGW',
 'FAKLLAKALKKLL',
 'FAKLFAKAFKKAL',
 'WFKKIPKFLHLLKKF',
 'FLPLLISALTSLFPKLGK',
 'IPPFIKKVLTTVF',
 'FKRLAKIKVLRLAKIKR',
 'KLKKLFKKILKY',
 'FAKLLKLAAKKLL',
 'AAWKWAWAK

In [6]:
# ABCP_tokenisation
with open("Non_ABCP_train.fasta") as non_abcp:
    non_abcp_lines = [line.strip("\n") for line in non_abcp.readlines()[1::2]]

non_abcp_lines = [l for l in non_abcp_lines if len(l) <51]
non_abcp_lines

['RILFDWYPTSDSTDPVEMRLFLRCQG',
 'GIINTLQKYYCRVRGGRCAVLSCLPKEEQIGKCSTRGRKCCRRKK',
 'IYSFDGRDIMTDPSWPQKVIWHGSSPHGVRLVDNYCEAWRTA',
 'SNSSKKTTNKEQSDKSAESRMATSGIVESFPTGALVPKAETGVLNFLQK',
 'HILTSKTRKNKRNLRKGGIVAASDHKNISCLIPYKMPKMKTHRGSAKR',
 'LFEDTNLCAIHAKRVTIQSKDIQLARRLRGERSMARTKQ',
 'PKIQKLLQDFFNGKELNKSINPDEAVAYGAAVQ',
 'RVAQALPEVDVTGLDLETVARDVLRHPTVASKS',
 'GPVHPGPLSPEPQPMGVRVICGHC',
 'YFYPDNPKAYQISQFDQPIGEN',
 'MGYRSSMPIIELKLTGPASEQQAMEKLWLDV',
 'DFKDWMKTAGEWLKKKGPGILKAAMAAAT',
 'HTHQDFQPVLHLVALNTPLSGGMRGIR',
 'LGDAIKHLDDLKGTFAQLSELHCDKLHVDPENF',
 'LTQRLRKTTDGQIQHHLLREVFDAPVDEE',
 'GGPHLEYFVRCEVPSADLPALLSSVRRVAEDVRGAGENKVLWFPRKVS',
 'QHLVTTKILYRYASKSKFNYDLSSNKIFHEPYPQDRSN',
 'SIGEQQMVEIAKALSFESKVIIMDE',
 'AKELRPIVEKLVTLGKKGGLALRRQAISEMKDQDQVRKLFDVLAKR',
 'NYPLSILAADKKTDLLTFFQSEDELTADIFYT',
 'AGDGTTTATVLAQALVKEGLRNVAAGAAPSALKHGIEVSVEAIA',
 'RQKNHGIHFRVLAKALRMSGGDHIHSGTVV',
 'MQIITEPCQMQAVVEKLRLNRQLIGVVMTMGALHEGH',
 'DIQIPGIKKPTHRDIIIPNWNPNVRTQPWQRFGGNKS',
 'TEADRILSDAKAQADRMVSEARQHSERMVAD

### generating training embeddings for ABCP and Non-ABCP

In [9]:
# Define models
model_names = {
        'ProtBERT': 'Rostlab/prot_bert_bfd',
        'ESM-2': 'facebook/esm2_t33_650M_UR50D'
    }

# running embeddings for ABCPs
abcp_embedding_list = []
# --- Generate embeddings for each model ---
for model_alias, model_id in model_names.items():
        print(f"\n--- Generating embeddings with {model_alias} ---")
        embeddings = get_peptide_embeddings(abcp_lines, model_id, device="cpu")
        abcp_embedding_list.append(embeddings)
        print(f"Shape of embeddings for {model_alias}: {embeddings.shape}")
        print(f"Example embedding for the first peptide ('{abcp_lines[0]}'):")
        print(embeddings[0][:5]) # Print the first 5 dimensions as a sampl


--- Generating embeddings with ProtBERT ---
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=

Generating embeddings with Rostlab/prot_bert_bfd: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 240/240 [00:44<00:00,  5.42it/s]


Shape of embeddings for ProtBERT: (240, 1024)
Example embedding for the first peptide ('KAAKKAWKWAKKAAKWAKKAA'):
[-0.03495022 -0.1409148   0.01816412  0.11302471 -0.03934536]

--- Generating embeddings with ESM-2 ---


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): EsmOut

Generating embeddings with facebook/esm2_t33_650M_UR50D: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 240/240 [01:57<00:00,  2.05it/s]


Shape of embeddings for ESM-2: (240, 1280)
Example embedding for the first peptide ('KAAKKAWKWAKKAAKWAKKAA'):
[-0.06342817  0.05793704 -0.16856852 -0.00843214  0.17256515]


In [10]:
# Define models
model_names = {
        'ProtBERT': 'Rostlab/prot_bert_bfd',
        'ESM-2': 'facebook/esm2_t33_650M_UR50D'
    }
# running embeddings for ABCPs
non_abcp_embedding_list = []
# --- Generate embeddings for each model ---
for model_alias, model_id in model_names.items():
        print(f"\n--- Generating embeddings with {model_alias} ---")
        embeddings = get_peptide_embeddings(non_abcp_lines, model_id, device = "cpu")
        non_abcp_embedding_list.append(embeddings)
        print(f"Shape of embeddings for {model_alias}: {embeddings.shape}")
        print(f"Example embedding for the first peptide ('{non_abcp_lines[0]}'):")
        print(embeddings[0][:5]) # Print the first 5 dimensions as a sampl


--- Generating embeddings with ProtBERT ---
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=

Generating embeddings with Rostlab/prot_bert_bfd: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 467/467 [01:36<00:00,  4.86it/s]


Shape of embeddings for ProtBERT: (467, 1024)
Example embedding for the first peptide ('RILFDWYPTSDSTDPVEMRLFLRCQG'):
[ 2.3475490e-05  1.6874263e-02  1.0513637e-02 -9.3769077e-03
  1.3947217e-02]

--- Generating embeddings with ESM-2 ---


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): EsmOut

Generating embeddings with facebook/esm2_t33_650M_UR50D: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 467/467 [02:43<00:00,  2.86it/s]

Shape of embeddings for ESM-2: (467, 1280)
Example embedding for the first peptide ('RILFDWYPTSDSTDPVEMRLFLRCQG'):
[ 0.07686592  0.0578908   0.06237853  0.07385929 -0.03586654]


In [13]:
import pandas as pd

abcp_embed_ProtBERT = abcp_embedding_list[0]
abcp_embed_ESM2 = abcp_embedding_list[1]


non_abcp_embed_ProtBERT = non_abcp_embedding_list[0]
non_abcp_embed_ESM2 = non_abcp_embedding_list[1]


df_abcp_ProtBERT = pd.DataFrame(abcp_embed_ProtBERT)
df_abcp_ESM2 = pd.DataFrame(abcp_embed_ESM2)

df_non_abcp_ProtBERT = pd.DataFrame(non_abcp_embed_ProtBERT)
df_non_abcp_ESM2 = pd.DataFrame(non_abcp_embed_ESM2)

df_abcp_ProtBERT["Target"] = 1
df_abcp_ESM2["Target"] = 1

df_non_abcp_ProtBERT["Target"] = 0
df_non_abcp_ESM2["Target"] = 0

df_ProtBERT = pd.concat([df_abcp_ProtBERT,df_non_abcp_ProtBERT])
df_ESM2 = pd.concat([df_abcp_ESM2,df_non_abcp_ESM2])

df_ProtBERT.to_csv("ProtBERT_embeddings_training.csv",index=False)
df_ESM2.to_csv("ESM2_embeddings_training.csv",index=False)

In [14]:
#generating test data

In [15]:
# ABCP_tokenisation
with open("ABCP_test.fasta") as abcp:
    abcp_lines = [line.strip("\n") for line in abcp.readlines()[1::2]]

abcp_lines


# ABCP_tokenisation
with open("Non_ABCP_test.fasta") as non_abcp:
    non_abcp_lines = [line.strip("\n") for line in non_abcp.readlines()[1::2]]

non_abcp_lines = [l for l in non_abcp_lines if len(l) <51]
non_abcp_lines


# Define models
model_names = {
        'ProtBERT': 'Rostlab/prot_bert_bfd',
        'ESM-2': 'facebook/esm2_t33_650M_UR50D'
    }

# running embeddings for ABCPs
abcp_embedding_list = []
# --- Generate embeddings for each model ---
for model_alias, model_id in model_names.items():
        print(f"\n--- Generating embeddings with {model_alias} ---")
        embeddings = get_peptide_embeddings(abcp_lines, model_id, device="cpu")
        abcp_embedding_list.append(embeddings)
        print(f"Shape of embeddings for {model_alias}: {embeddings.shape}")
        print(f"Example embedding for the first peptide ('{abcp_lines[0]}'):")
        print(embeddings[0][:5]) # Print the first 5 dimensions as a sampl


# Define models
model_names = {
        'ProtBERT': 'Rostlab/prot_bert_bfd',
        'ESM-2': 'facebook/esm2_t33_650M_UR50D'
    }
# running embeddings for ABCPs
non_abcp_embedding_list = []
# --- Generate embeddings for each model ---
for model_alias, model_id in model_names.items():
        print(f"\n--- Generating embeddings with {model_alias} ---")
        embeddings = get_peptide_embeddings(non_abcp_lines, model_id, device = "cpu")
        non_abcp_embedding_list.append(embeddings)
        print(f"Shape of embeddings for {model_alias}: {embeddings.shape}")
        print(f"Example embedding for the first peptide ('{non_abcp_lines[0]}'):")
        print(embeddings[0][:5]) # Print the first 5 dimensions as a sampl



abcp_embed_ProtBERT = abcp_embedding_list[0]
abcp_embed_ESM2 = abcp_embedding_list[1]


non_abcp_embed_ProtBERT = non_abcp_embedding_list[0]
non_abcp_embed_ESM2 = non_abcp_embedding_list[1]


df_abcp_ProtBERT = pd.DataFrame(abcp_embed_ProtBERT)
df_abcp_ESM2 = pd.DataFrame(abcp_embed_ESM2)

df_non_abcp_ProtBERT = pd.DataFrame(non_abcp_embed_ProtBERT)
df_non_abcp_ESM2 = pd.DataFrame(non_abcp_embed_ESM2)

df_abcp_ProtBERT["Target"] = 1
df_abcp_ESM2["Target"] = 1

df_non_abcp_ProtBERT["Target"] = 0
df_non_abcp_ESM2["Target"] = 0

df_ProtBERT = pd.concat([df_abcp_ProtBERT,df_non_abcp_ProtBERT])
df_ESM2 = pd.concat([df_abcp_ESM2,df_non_abcp_ESM2])

df_ProtBERT.to_csv("ProtBERT_embeddings_testing.csv",index=False)
df_ESM2.to_csv("ESM2_embeddings_testing.csv",index=False)


--- Generating embeddings with ProtBERT ---
BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=

Generating embeddings with Rostlab/prot_bert_bfd: 100%|███████████████| 61/61 [00:15<00:00,  3.89it/s]


Shape of embeddings for ProtBERT: (61, 1024)
Example embedding for the first peptide ('THPPTTTTTTTTTTTTTAAPATTT'):
[ 0.04362527  0.03401236 -0.07515986 -0.19274849  0.00436258]

--- Generating embeddings with ESM-2 ---


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): EsmOut

...

Shape of embeddings for ProtBERT: (117, 1024)
Example embedding for the first peptide ('AVRIGPCDQVCPRIVPERHECCRAHGRSGYAYCSGGGMYCN'):
[0.0343178  0.010809   0.01444821 0.03730674 0.0705734 ]

--- Generating embeddings with ESM-2 ---


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): EsmOut

Generating embeddings with facebook/esm2_t33_650M_UR50D: 100%|██████| 117/117 [00:40<00:00,  2.90it/s]


Shape of embeddings for ESM-2: (117, 1280)
Example embedding for the first peptide ('AVRIGPCDQVCPRIVPERHECCRAHGRSGYAYCSGGGMYCN'):
[-0.03034343 -0.05316237 -0.00347624  0.0399634  -0.00025339]
